In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv(r"C:\Users\ganesh\Desktop\All Projects\Predicative care load Forecasting\data\processed\hhs_eda_dataset.csv")

In [3]:
df["Date"]=pd.to_datetime(df["Date"])
df=df.sort_values("Date").reset_index(drop=True)

In [4]:
print(df.shape)
print(df.head())

(720, 11)
        Date  Children apprehended and placed in CBP custody*  \
0 2023-01-12                                             33.0   
1 2023-01-22                                             32.0   
2 2023-01-23                                             32.0   
3 2023-01-24                                             47.0   
4 2023-01-25                                             20.0   

   Children in CBP custody  Children transferred out of CBP custody  \
0                     53.0                                     34.0   
1                     49.0                                     39.0   
2                     50.0                                     39.0   
3                     42.0                                     47.0   
4                     22.0                                     41.0   

   Children in HHS Care  Children discharged from HHS Care  Month  HHS_7D_MA  \
0                  6566                              436.0      1        NaN   
1           

In [5]:
model_df=df[
    [
        "Date",
        "Children apprehended and placed in CBP custody*",
        "Children in CBP custody",
        "Children transferred out of CBP custody",
        "Children in HHS Care",
        "Children discharged from HHS Care"   
    ]
].copy()

In [6]:
model_df.columns.tolist()

['Date',
 'Children apprehended and placed in CBP custody*',
 'Children in CBP custody',
 'Children transferred out of CBP custody',
 'Children in HHS Care',
 'Children discharged from HHS Care']

In [7]:
model_df["day_of_week"]= model_df["Date"].dt.dayofweek
model_df["day_of_month"]= model_df["Date"].dt.day
model_df["month"]= model_df["Date"].dt.month
model_df["quarter"]= model_df["Date"].dt.quarter
model_df["year"]= model_df["Date"].dt.year

model_df["week_of_year"]= (
    model_df["Date"].dt.isocalendar().week.astype(int)
)

model_df["is_weekend"]= (
    model_df["day_of_week"] >= 5
).astype(int)

In [8]:
model_df[
    [
        "Date",
        "day_of_week",
        "month",
        "quarter",
        "year",
        "week_of_year",
        "is_weekend"
    ]
].head(10)

,Date,day_of_week,month,quarter,year,week_of_year,is_weekend
0,2023-01-12,3,1,1,2023,2,0
1,2023-01-22,6,1,1,2023,3,1
2,2023-01-23,0,1,1,2023,4,0
3,2023-01-24,1,1,1,2023,4,0
4,2023-01-25,2,1,1,2023,4,0
5,2023-01-29,6,1,1,2023,4,1
6,2023-01-30,0,1,1,2023,5,0
7,2023-01-31,1,1,1,2023,5,0
8,2023-02-01,2,2,1,2023,5,0
9,2023-02-02,3,2,1,2023,5,0


In [9]:
model_df["Net_Pressure"]= (
    model_df["Children transferred out of CBP custody"] 
    - model_df["Children discharged from HHS Care"]
)
model_df["Net_Pressure"].head(5)

0   -402.0
1   -188.0
2   -142.0
3   -128.0
4   -139.0
Name: Net_Pressure, dtype: float64

In [10]:
model_df["Discharge_to_Transfer_Ratio"]= (
    model_df["Children discharged from HHS Care"]
    / model_df["Children transferred out of CBP custody"].replace(0,np.nan)
)
model_df["Discharge_to_Transfer_Ratio"].head(5)

0    12.823529
1     5.820513
2     4.641026
3     3.723404
4     4.390244
Name: Discharge_to_Transfer_Ratio, dtype: float64

In [11]:
hhs_series= (
    model_df.set_index("Date")["Children in HHS Care"]
)

In [12]:
model_df["hhs_lag_1"]= (
    model_df["Date"].map(hhs_series.shift(freq="1D"))
)

In [13]:
model_df["hhs_lag_7"]= (
    model_df["Date"].map(hhs_series.shift(freq="7D"))
)

In [14]:
model_df["hhs_lag_14"]= (
    model_df["Date"].map(hhs_series.shift(freq="14D"))
)

In [15]:
model_df["hhs_lag_28"]= (
    model_df["Date"].map(hhs_series.shift(freq="28D"))
)

In [16]:
transfer_series= (
    model_df.set_index("Date")["Children transferred out of CBP custody"]
)

model_df["transfer_lag_1"]= (
    model_df["Date"].map(transfer_series.shift(freq="1D"))
)

model_df["transfer_lag_7"]= (
    model_df["Date"].map(transfer_series.shift(freq="7D"))
)

model_df["transfer_lag_14"]= (
    model_df["Date"].map(transfer_series.shift(freq="14D"))
)

In [17]:
discharge_series= (
    model_df.set_index("Date")["Children discharged from HHS Care"]
)

model_df["discharge_lag_1"]= (
    model_df["Date"].map(discharge_series.shift(freq="1D"))
)

model_df["discharge_lag_7"]= (
    model_df["Date"].map(discharge_series.shift(freq="7D"))
)

model_df["discharge_lag_14"]= (
    model_df["Date"].map(discharge_series.shift(freq="14D"))
)

In [18]:
rolling_df= model_df.set_index("Date").sort_index()

In [19]:
rolling_df["hhs_rolling_mean_7d"]= (
    rolling_df["Children in HHS Care"].rolling("7D").mean()
)

rolling_df["hhs_rolling_mean_14d"]= (
    rolling_df["Children in HHS Care"].rolling("14D").mean()
)

In [20]:
rolling_df["hhs_rolling_std_7d"]= (
    rolling_df["Children in HHS Care"].rolling("7D").std()
)

rolling_df["hhs_rolling_std_14d"]= (
    rolling_df["Children in HHS Care"].rolling("14D").std()
)

In [21]:
model_df= rolling_df.reset_index()

In [22]:
model_df.shape

(720, 29)

In [23]:
model_df.columns.tolist()

['Date',
 'Children apprehended and placed in CBP custody*',
 'Children in CBP custody',
 'Children transferred out of CBP custody',
 'Children in HHS Care',
 'Children discharged from HHS Care',
 'day_of_week',
 'day_of_month',
 'month',
 'quarter',
 'year',
 'week_of_year',
 'is_weekend',
 'Net_Pressure',
 'Discharge_to_Transfer_Ratio',
 'hhs_lag_1',
 'hhs_lag_7',
 'hhs_lag_14',
 'hhs_lag_28',
 'transfer_lag_1',
 'transfer_lag_7',
 'transfer_lag_14',
 'discharge_lag_1',
 'discharge_lag_7',
 'discharge_lag_14',
 'hhs_rolling_mean_7d',
 'hhs_rolling_mean_14d',
 'hhs_rolling_std_7d',
 'hhs_rolling_std_14d']

In [24]:
model_df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,day_of_week,day_of_month,month,quarter,...,transfer_lag_1,transfer_lag_7,transfer_lag_14,discharge_lag_1,discharge_lag_7,discharge_lag_14,hhs_rolling_mean_7d,hhs_rolling_mean_14d,hhs_rolling_std_7d,hhs_rolling_std_14d
0,2023-01-12,33.0,53.0,34.0,6566,436.0,3,12,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,6566.000000,6566.000000,NaN,NaN
1,2023-01-22,32.0,49.0,39.0,7122,227.0,6,22,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,7122.000000,6844.000000,NaN,393.151370
2,2023-01-23,32.0,50.0,39.0,7280,181.0,0,23,1,1,...,39.0,NaN,NaN,227.0,NaN,NaN,7201.000000,6989.333333,111.722871,375.032443
3,2023-01-24,47.0,42.0,47.0,7433,175.0,1,24,1,1,...,39.0,NaN,NaN,181.0,NaN,NaN,7278.333333,7100.250000,155.506699,378.122004
4,2023-01-25,20.0,22.0,41.0,7538,180.0,2,25,1,1,...,47.0,NaN,NaN,175.0,NaN,NaN,7343.250000,7187.800000,181.599146,381.519593


In [25]:
missing_summary= (
    model_df.isnull().sum().sort_values(ascending=False)
)

missing_summary

hhs_lag_1                                          162
transfer_lag_1                                     162
discharge_lag_1                                    162
hhs_lag_28                                          64
transfer_lag_14                                     54
discharge_lag_14                                    54
hhs_lag_14                                          54
hhs_lag_7                                           47
transfer_lag_7                                      47
discharge_lag_7                                     47
hhs_rolling_std_7d                                   3
Discharge_to_Transfer_Ratio                          3
hhs_rolling_std_14d                                  1
is_weekend                                           0
week_of_year                                         0
year                                                 0
quarter                                              0
month                                                0
day_of_mon

In [26]:
target = "Children in HHS Care"
print(model_df[target].describe())
print(model_df[target].isna().sum())

count      720.000000
mean      6061.275000
std       2833.070109
min       1972.000000
25%       2467.750000
50%       6406.500000
75%       8010.250000
max      11516.000000
Name: Children in HHS Care, dtype: float64
0


In [27]:
model_df.to_csv(
    r"C:\Users\ganesh\Desktop\All Projects\Predicative care load Forecasting\data\processed\hhs_feature.csv",
)